
# Perturbation analysis with pertpy

A Cell Painting screen *is* a perturbation experiment, so this is the most natural fit of
the three notebooks — and also the one where the sample-size assumptions differ most.

**What `pertpy` was built for, and what we have:**

| assumption | perturb-seq / sci-Plex | EU-OS `IMTM_HepG2` |
| --- | --- | --- |
| observations per perturbation | hundreds to thousands of cells | **4 wells** |
| explicit replicate structure | usually none | **4 replicate plates** |
| dose series | sci-Plex: 4+ doses | **none** — single 10 uM dose |
| control | non-targeting gRNA / vehicle | 784 DMSO wells |
| readout | counts, needs a count model | continuous z-scores, no count model needed |

**The n = 4 problem is the central caveat of this notebook.** Distribution-based metrics —
E-distance, MMD, Wasserstein — estimate and compare *distributions*. With four points per
compound in 50 dimensions they are computable but high-variance. Mitigations, used below:
score against the 784 DMSO wells (a well-estimated reference), work in PCA space rather
than 2,776 features, bootstrap, and lean on **replicate reproducibility** — the metric the
field actually uses, which turns the small *n* from a weakness into the measurement itself.

## 1. Setup

In [ ]:
import os
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc

sc.settings.verbosity = 1
sc.set_figure_params(dpi=90, frameon=False, figsize=(4, 4))

# Override with EU_OS_H5AD if your copy lives elsewhere.
H5AD = Path(os.environ.get("EU_OS_H5AD", "../data/eu_os_imtm_hepg2.h5ad"))
CLIP = 10.0  # see the heavy-tail section of notebook 01

import pertpy as pt

# Notebook 01, section 13 writes this with PCA, both Harmony embeddings, UMAP and Leiden
# labels already computed. REP is the embedding notebook 01 settled on: everything here
# that works on an embedding reads it from uns, so changing BATCH_KEY in notebook 01
# section 8b changes the results in this notebook.
INTEGRATED = H5AD.with_name("eu_os_imtm_hepg2_integrated.h5ad")
FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

if INTEGRATED.exists():
    adata = ad.read_h5ad(INTEGRATED)
    REP = adata.uns["integration"]["rep"]
    print(f"loaded {INTEGRATED.name}")
    print(f"  batch_key = {adata.uns['integration']['batch_key']!r}  ->  REP = {REP!r}")
else:
    # Standalone fallback: redo notebook 01's clipping and PCA, no batch correction.
    print(f"{INTEGRATED.name} not found -- run notebook 01 to build it. "
          f"Falling back to {H5AD.name}.")
    adata = ad.read_h5ad(H5AD)
    adata.layers["robust_z"] = adata.X.copy()
    adata.X = np.clip(adata.X, -CLIP, CLIP)          # notebook 01, section 4
    sc.pp.pca(adata, n_comps=50, zero_center=True, svd_solver="arpack")
    REP = "X_pca"

adata


## 2. There is no dose-response here

Worth checking before planning any dose analysis, because `concentration_uM` has three
distinct values and looks like a dose series until you cross it with `pert_type`.

> **Which representation, and why.** Every distance in this notebook — E-distance,
> `DistanceTest`, the replicate-similarity metric in section 5, the induction score in
> section 7 — is computed on an *embedding*, not on features. That makes this the one place
> in the series where notebook 01's batch-correction decision actually bites, so all of
> them read `REP` from `adata.uns['integration']` rather than hard-coding `X_pca`.
>
> Notebook 01 settled on `BATCH_KEY = "Replicate"`, so `REP` is `X_pca_harmony_replicate`
> and every number quoted in this notebook's text is for that. The correction is close to a
> no-op — `Replicate` is the physical repeat, and per-plate `mad_robustize` had already
> removed most of its effect — so section 5 reads **65.2%** against the position-matched
> null, a shade above the **64.3%** on the uncorrected `X_pca`. Harmonizing `Plate` instead
> would pull it down to **56.9%**, which is precisely why notebook 01 did not.
>
> (Notebook 01, section 8b reports those same three conditions as 65.7%, 66.0% and 60.1%.
> It resamples the null from a different point in the RNG stream, so the absolute values
> differ by about a point between the two notebooks. Read the difference between
> conditions, not the absolute number.)
>
> The per-feature tests (`WilcoxonTest` in section 4, the pseudobulk profiles in section 6)
> still use `.X`, for the same reason notebook 02 does: there is no route from 50 corrected
> PCs back to 2,776 named features.

In [ ]:

print(pd.crosstab(adata.obs.concentration_uM, adata.obs.pert_type))


The three concentrations are perfectly collinear with perturbation class: 10 uM is every
treatment, 0 is DMSO, 5 uM is the positive controls. **Concentration carries no
information beyond `pert_type`**, so `pt.tl.Dialogue`-style dose modelling, EC50 fitting
and dose-trajectory analysis are all impossible on this subset — as they are on the whole
EU-OS bioactives study, which is a single-dose screen by design.

> **If you need dose-response.** Two options on this machine:
>
> - **EUbOPEN** (`/home/lheumos/data/2024_EUbOPEN_CellPainting/eubopen.h5ad`) — 39,206
>   wells x 4,943 features with **10 concentration levels** in `obs.concentration`, plus
>   per-well cell counts. This is the right dataset for dose-response.
> - **CPJUMP1** — no dose series, but 4 timepoints and 3 perturbation modalities, so it
>   supports time-course analysis instead.
>
> **Surrogate axis within this dataset.** `cell_count` is a continuous, monotone measure
> of how hard a compound hit the cells. Treating it as a severity axis (rather than a dose
> axis) lets you ask which morphological changes track general toxicity and which are
> specific — useful precisely because notebook 01 showed cell count dominates PC1.

In [ ]:

# Sketch of the dose-response analysis, for when you switch to EUbOPEN.
# eub = ad.read_h5ad("/home/lheumos/data/2024_EUbOPEN_CellPainting/eubopen.h5ad")
# print(eub.obs.concentration.value_counts().sort_index())
# dist = pt.tl.Distance(metric="edistance", obsm_key=REP)
# per_dose = {
#     c: dist.onesided_distances(eub[eub.obs.concentration.isin([0, c])],
#                                groupby="id_perturbator", selected_group="DMSO")
#     for c in sorted(eub.obs.concentration.unique())
# }
# # -> distance from control as a function of dose = a morphological dose-response curve


## 3. Effect size: how far did each compound move from DMSO?

This is the "induction" or "activity" score of a screen, and it is the direct analogue of
perturbation effect size in perturb-seq.

> **Assumption.** `pt.tl.Distance` metrics compare two *samples of cells*. `edistance` is
> an energy statistic over pairwise distances, so it scales with both group size and
> dimensionality — distances are comparable across groups of similar *n*, not across
> arbitrary ones.
>
> **Cell Painting.** Compute in **PCA space** (`obsm_key=REP`), not on 2,776 raw
> features, and only compare compounds with the same replicate count. With n = 4 the
> estimate is noisy; use `bootstrap=True` when you need an interval.

In [ ]:

# Start on a fast, interpretable subset: controls + annotated tubulin binders.
mask = adata.obs.pert_type.isin(["negcon", "poscon"]) | adata.obs.tubulin_binder
sub = adata[mask].copy()

# The positive controls have no `pdid`, so `compound` is an empty string for them --
# fall back to `EOS` (which holds "Nocodazole" / "Tetrandrine") or they all collapse
# into one nameless group.
compound = sub.obs.compound.astype(str)
sub.obs["group"] = np.where(
    sub.obs.pert_type.eq("negcon"),
    "DMSO",
    np.where(compound.str.len() > 0, compound, sub.obs.EOS.astype(str)),
)

dist = pt.tl.Distance(metric="edistance", obsm_key=REP)
edist = dist.onesided_distances(sub, groupby="group", selected_group="DMSO", show_progressbar=False)
print(edist.sort_values(ascending=False).head(14).round(1).to_string())


Two things to take from this ranking.

**The assay works.** Nocodazole, colchicine, mebendazole, podofilox and paclitaxel — real
microtubule poisons — all land near the top, far above DMSO's internal spread. So does
`Tetrandrine`, one of the screen's two designated positive controls, which is the cleanest
possible confirmation: the wells the experimenters chose as "this should show an effect"
do show one.

**The annotation is noisy.** Lapatinib, pelitinib and canertinib usually top the list, and
they are EGFR/ERBB inhibitors that happen to carry the `tubulin_binder` flag in this
export. They produce large morphological changes, but not because of tubulin. Treat
`tubulin_binder` as a useful-but-imperfect positive control set, and never as a label to
train on without inspection.

> **Scaling up.** Over all 2,459 compounds `onesided_distances` is O(n_groups) pairwise
> computations and takes a while. Options: use `metric="mse"` or `"pearson_distance"`
> (mean-based, far cheaper, and better behaved at n = 4 since they compare centroids
> rather than distributions); or aggregate to consensus profiles first (section 5) and use
> a plain distance.

In [ ]:

# TODO: run over the full library. Mean-based metrics are the pragmatic choice at n=4.
# adata.obs["group"] = np.where(adata.obs.pert_type.eq("negcon"), "DMSO", adata.obs.EOS.astype(str))
# cheap = pt.tl.Distance(metric="mse", obsm_key=REP)
# activity = cheap.onesided_distances(adata, groupby="group", selected_group="DMSO")
# activity.sort_values(ascending=False).head()


## 4. Is the effect significant?

Section 3 gave effect sizes. To turn those into p-values, permute the perturbation labels,
recompute the distance, and see how often chance beats the observed value.

**Mind the class names — this one is a trap.** `pt.tl.PermutationTest` sounds like the tool
for this job, but in pertpy it is a **differential-expression** method, a sibling of
`TTest` and `WilcoxonTest`, that permutes a per-**feature** test statistic. The tool that
permutes a *distance between groups* is **`pt.tl.DistanceTest`**. They answer different
questions: "did this feature change" versus "did this perturbation do anything at all".
We want the latter here, and come back to the former below.

> **Validated against `pertpy 1.3.0`** (`DistanceTest` lives in
> `pertpy.tools._distances._distance_tests`). Check `importlib.metadata.version("pertpy")`
> if a call below fails — this part of the API has moved between releases.

> **Assumption.** Permutation testing assumes labels are exchangeable under the null.
>
> **Cell Painting.** They are *not* exchangeable across plates: plate and well position
> carry real signal (notebook 01), so an unrestricted permutation mixes wells that differ
> systematically for non-biological reasons and gives an optimistic p-value. `DistanceTest`
> permutes globally, so read these p-values as an upper bound on the evidence. Permuting
> within plate would be stricter, and is not exposed by the current API.

In [ ]:

import importlib.metadata as md

print("pertpy", md.version("pertpy"))

# n_perms sets a floor on the p-value: the smallest attainable value is ~1/n_perms.
# With 23 groups and Holm-Sidak correction you need p < ~0.05/22 = 0.0023 to stay
# significant, so n_perms=100 (floor 0.01) cannot ever get there -- see below.
dtest = pt.tl.DistanceTest(
    "edistance",
    n_perms=1000,
    obsm_key=REP,       # the embedding notebook 01 chose, not necessarily X_pca
    alpha=0.05,
    correction="holm-sidak",
)
result = dtest.test_precomputed(sub, groupby="group", contrast="DMSO", verbose=False)

print(f"\nsignificant after correction: {int(result.significant_adj.sum())} of {len(result)}")
print(result.sort_values("distance", ascending=False).head(8).round(4).to_string())
print("\nnot significant:")
print(result[~result.significant_adj].round(4).to_string())


Three things worth taking from that output.

**22 of 23 groups are significant, and the exception is `DMSO` itself** — tested against
itself it has distance exactly 0 and p = 1.0. That is the sanity check passing, not a
finding.

**The permutation count and the multiple-testing correction interact.** `n_perms` sets a
floor on the achievable p-value of roughly `1/n_perms`. At `n_perms=100` every group
bottoms out at p = 0.01, and Holm-Šidák across 22 comparisons then needs about 0.0023 —
unreachable. Run it and you get **zero significant groups, with the smallest adjusted
p-value at 0.21**, no matter how large the effects are. `n_perms=1000` (floor 0.001) fixes
it. The rule of thumb: `n_perms` should comfortably exceed `n_groups / alpha`. This is a
pure artifact of the method and is very easy to misread as a biological null result.

**`DistanceTest` and `Distance` do not return identical distances.** Compare the
`distance` column here with `edist` from section 3: they correlate at r = 0.996 but differ
by up to ~52 units, and the gap tracks group size — 10 to 52 for the 4-well compounds,
~1.8 for the two 56-well control groups, and exactly 0 for the 784-well DMSO group.
`test_precomputed` reaches the statistic through a precomputed pairwise-distance matrix
while `Distance.onesided_distances` computes it directly, and the two normalise the
within-group term differently. At n = 4 that matters.

> **So don't mix them.** Quote effect sizes and p-values from the *same* call, or say which
> came from which. The ranking is essentially unaffected, which is why this is a
> reporting-hygiene issue rather than a correctness one.


### Scaling up, and the per-feature question

`test_precomputed` on the 23-group subset takes about 17 s at `n_perms=1000`. The full
library is 2,460 groups, so the same call is far more expensive and needs a
10,556 x 10,556 distance matrix — subsample compounds, drop `n_perms`, or (better) call
hits with the cheaper induction-plus-reproducibility combination in sections 5-7 and
reserve `DistanceTest` for the shortlist.

For "**which features** changed", use a per-feature test — and here is where
`pt.tl.PermutationTest` would come in, except that its default test statistic is
`log2(mean(x1)) - log2(mean(x0))`. On signed z-scores the mean is frequently negative, so
that returns NaN for **58% of features**, and the surviving p-values are degenerate. You
can pass a valid `test_statistic` callable to rescue it, but there is no reason to: a
rank-based test makes no positivity assumption and is far faster.

In [ ]:

# One binary contrast: the 784 DMSO wells vs the 80 annotated tubulin-binder wells.
binary = sub[sub.obs.pert_type.ne("poscon")].copy()
binary.obs["contrast"] = np.where(binary.obs.pert_type.eq("negcon"), "DMSO", "tubulin")

wilcox = pt.tl.WilcoxonTest.compare_groups(
    binary, column="contrast", baseline="DMSO", groups_to_compare="tubulin"
)
wilcox = wilcox.merge(adata.var[["family", "channel"]], left_on="variable", right_index=True)
print(f"{len(wilcox)} features tested, {(wilcox.adj_p_value < 0.05).sum()} significant at adj p < 0.05")
print("\nfamily composition of the 200 most significant features:")
print(wilcox.nsmallest(200, "adj_p_value").family.value_counts().to_string())


`RadialDistribution` and `Neighbors` dominate the top of that list — organelle signal
redistributing outward, and cells changing how much they touch. Both are what you would
expect from mitotic arrest and cytoskeletal collapse, and both agree with the programme
scores in notebook 02, which is a useful cross-method check.

> **Ignore the p-values as a ranking device.** With 784 control wells the test is so
> powerful that ~2,150 of 2,776 features come out significant, and hundreds are pinned at
> adj p = 0. Significance here tells you almost nothing; **rank by effect size** (the
> `statistic` column, or a difference in medians) and use the p-value only as a filter.
> This is the same trap as calling every gene differentially expressed in a large scRNA
> dataset, just more extreme because n is large and the features are heavily correlated.


## 5. Replicate reproducibility — what screens actually use

This has no single-cell counterpart and is the most important metric in image-based
profiling. The logic: at a single dose most compounds do nothing, so "did it move?" is a
weak question. The strong question is **"did it move the same way every time?"** If a
compound's 4 replicate wells resemble each other more than they resemble random wells, the
phenotype is real and reproducible.

This is variously called *percent replicating*, *fraction retrieved*, or scored by **mean
average precision (mAP)**. `pertpy` does not implement it, so here it is directly.

In [ ]:

from sklearn.preprocessing import normalize

trt = adata[adata.obs.pert_type.eq("trt")].copy()
P = normalize(trt.obsm[REP])              # cosine similarity via dot products
eos = trt.obs.EOS.astype(str).to_numpy()

rng = np.random.default_rng(0)
upper = np.triu_indices(4, k=1)
positions = pd.Series(np.arange(len(eos)), index=eos)

records = []
for compound, idx in positions.groupby(level=0):
    idx = idx.to_numpy()
    if len(idx) != 4:
        continue
    rnd = rng.choice(len(eos), size=4, replace=False)
    records.append({
        "EOS": compound,
        "same": (P[idx] @ P[idx].T)[upper].mean(),
        "null": (P[rnd] @ P[rnd].T)[upper].mean(),
    })

rep = pd.DataFrame(records).set_index("EOS")
same, null = rep["same"].to_numpy(), rep["null"].to_numpy()
threshold = np.percentile(null, 95)
print(f"{len(rep)} compounds with 4 replicates")
print(f"mean within-compound similarity: {same.mean():+.3f}   random quadruples: {null.mean():+.3f}")
print(f"95th percentile of null: {threshold:+.3f}")
print(f"percent replicating (above null 95th pct): {(same > threshold).mean():.1%}")

In [ ]:

fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(null, bins=60, alpha=0.6, label="random quadruples", density=True)
ax.hist(same, bins=60, alpha=0.6, label="same compound", density=True)
ax.axvline(threshold, color="k", ls="--", lw=1, label="null 95th pct")
ax.set_xlabel("mean pairwise cosine similarity"); ax.set_ylabel("density")
ax.legend(frameon=False); ax.set_frame_on(False)
fig.savefig(FIGDIR / "03_replicate_reproducibility.png", dpi=200, bbox_inches="tight")
plt.show()


### The null you choose changes the answer

That ~82% looks implausibly high for a single-dose bioactive screen, and the reason is the
null. Replicates of a compound do not differ only in compound — they sit at the **same well
coordinate on the same library plate**, differing solely by replicate. A fully random
quadruple shares none of that structure, so it is too easy a comparison.

Build nulls that hold the structure fixed and vary only the compound.

In [ ]:

well = trt.obs.Well.astype(str).to_numpy()
plate = trt.obs.Plate.astype(str).to_numpy()


def matched_null(match_on, n=2500):
    """Quadruples of *different* compounds that share a well coordinate or a plate."""
    out = []
    for _ in range(n):
        if match_on == "well":  # same coordinate, four different library plates
            cand = np.flatnonzero(well == rng.choice(np.unique(well)))
            seen, pick = set(), []
            for i in rng.permutation(cand):
                if plate[i] not in seen:
                    seen.add(plate[i])
                    pick.append(i)
                if len(pick) == 4:
                    break
            if len(pick) < 4:
                continue
            idx = np.asarray(pick)
        else:  # same library plate, four different wells
            cand = np.flatnonzero(plate == rng.choice(np.unique(plate)))
            idx = rng.choice(cand, size=4, replace=False)
        out.append((P[idx] @ P[idx].T)[upper].mean())
    return np.asarray(out)


nulls = {"random quadruple": null, "same well, diff plates": matched_null("well"),
         "same plate, diff wells": matched_null("plate")}

print(f"  {'same compound':<26} mean={same.mean():+.3f}  p95={np.percentile(same, 95):+.3f}")
for name, v in nulls.items():
    print(f"  {name:<26} mean={v.mean():+.3f}  p95={np.percentile(v, 95):+.3f}")
print()
for name, v in nulls.items():
    print(f"  percent replicating vs {name:<26} {(same > np.percentile(v, 95)).mean():6.1%}")

The measured picture:

| quadruple | mean similarity |
| --- | --- |
| same compound | **+0.61** |
| same well coordinate, different plates | +0.12 |
| same library plate, different wells | +0.08 |
| fully random | +0.04 |

Position **is** a real effect — same-coordinate wells are nearly three times as similar as
random ones — but it is nowhere near the compound effect. Percent replicating falls from
~82% against the random null to **~65%** against the position-matched null, with ~75%
against the plate-matched one. The headline number moves by 16 points depending on a choice
many papers do not state.

> **Report the position-matched figure.** It is the one that answers "did the *compound*
> reproduce", which is the question you meant to ask. And state which null you used.
>
> **A trap if you change `BATCH_KEY`.** These numbers are on `X_pca_harmony_replicate`,
> which leaves the plate effect in place, so the plate-matched null still means something.
> Set `BATCH_KEY = "Plate"` in notebook 01 and the same-plate null collapses from +0.08 to
> +0.05 — Harmony has removed the very thing the null was meant to hold fixed — and
> plate-matched percent replicating rises to ~77%, *above* the position-matched ~57%.
> Grading a plate correction with a plate-matched null is circular. The well-matched null
> is robust to that choice, which is another reason to prefer it.
>
> Two further refinements: published mAP uses **ranking** rather than mean similarity, so
> use [`copairs`](https://github.com/cytomining/copairs) if you need the standard metric;
> and note that 65% is still high for a bioactive library, partly because these compounds
> were preselected for activity and partly because reproducible *toxicity* counts as
> reproducible (see the `cell_count` confound in notebook 01).


## 6. Consensus profiles

Collapsing 4 wells to one profile per compound is the standard move before any
cross-compound comparison. It cuts noise, removes the n = 4 problem, and shrinks the
object from 10,668 x 2,776 to 2,459 x 2,776.

> **Assumption.** `pt.tl.PseudobulkSpace` defaults to `mode="sum"`, which is right for
> counts (summing UMIs across cells is a valid pseudobulk).
>
> **Cell Painting.** Summing z-scores is meaningless and scales with replicate count. Use
> `mode="median"` — robust to the one bad well per quadruplet that is entirely routine.

In [ ]:

ps = pt.tl.PseudobulkSpace()
consensus = ps.compute(adata, target_col="EOS", mode="median")
print(consensus)


## 7. Hit calling: combine induction with reproducibility

Neither axis alone is sufficient. A large effect that does not reproduce is an artifact
(one bad well); a reproducible but tiny effect is usually a position or plate quirk. Hits
are compounds that score on both.

In [ ]:

active = rep[["same"]].rename(columns={"same": "reproducibility"})

# Induction as distance from the DMSO centroid in PCA space (cheap, mean-based).
dmso_centroid = adata.obsm[REP][adata.obs.pert_type.eq("negcon").to_numpy()].mean(0)
d = np.linalg.norm(trt.obsm[REP] - dmso_centroid, axis=1)
active["induction"] = pd.Series(d, index=eos).groupby(level=0).median().reindex(active.index)

per_compound = adata.obs.assign(EOS=lambda f: f.EOS.astype(str)).drop_duplicates("EOS").set_index("EOS")
active["tubulin"] = per_compound.tubulin_binder.reindex(active.index).fillna(False)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(active.induction, active.reproducibility, s=4, alpha=0.25, label="library")
ax.scatter(active.induction[active.tubulin], active.reproducibility[active.tubulin],
           s=26, color="crimson", label="annotated tubulin binder")
ax.axhline(threshold, color="k", ls="--", lw=1)
ax.set_xlabel("induction (distance from DMSO centroid)")
ax.set_ylabel("replicate reproducibility")
ax.legend(frameon=False); ax.set_frame_on(False)
fig.savefig(FIGDIR / "03_induction_vs_reproducibility.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"hits (reproducible and in the top induction decile): "
      f"{((active.reproducibility > threshold) & (active.induction > active.induction.quantile(0.9))).sum()}")


Expect the tubulin binders to sit in the upper-right — high induction *and* high
reproducibility. If they do not, revisit the confound audit in notebook 01 before
trusting the rest of the plot.


## 8. Annotation: MOA and targets

`pertpy`'s metadata module fetches compound and mechanism annotation from public
databases, which is genuinely useful here: this dataset carries `smiles` and `inchikey`
for 2,457 compounds but only a single binary MOA flag (`tubulin_binder`).

In [ ]:

# Both of these download reference data on first use.
# moa = pt.md.Moa()
# moa.annotate(adata, query_id="compound", query_id_type="name")   # -> obs["moa"], obs["target"]
#
# compound = pt.md.Compound()
# compound.annotate_compounds(adata, query_id="compound")          # structure / properties from PubChem
print("MOA lookup available:", [m for m in dir(pt.md.Moa) if not m.startswith('_')])

In [ ]:

# pt.tl.Enrichment scores predefined target sets. Build them from obs.target_genes,
# which annotates ~95% of the library.
targets = (
    adata.obs.drop_duplicates("EOS").assign(EOS=lambda d: d.EOS.astype(str))
    .set_index("EOS")["target_genes"].astype(str)
)
target_sets = {}
for eos_id, genes in targets[targets.str.len() > 0].items():
    for gene in genes.split(";"):
        target_sets.setdefault(gene, []).append(eos_id)
target_sets = {g: c for g, c in target_sets.items() if len(c) >= 5}
print(f"{len(target_sets)} target genes with >=5 annotated compounds")
print(dict(list(sorted(target_sets.items(), key=lambda kv: -len(kv[1])))[:5]).keys())


> **How to use these sets.** `pt.tl.Enrichment.score` expects `targets` as
> `{set_name: [var_names]}` — i.e. the sets must index the **variable** axis. Our sets
> index compounds, so score them on the *transposed* consensus matrix (compounds as
> variables). The question it answers: do compounds sharing an annotated target share a
> morphological phenotype? A positive answer for a given gene is evidence that inhibiting
> it has a recognisable morphological signature, which is the basis for predicting targets
> of unannotated compounds.
>
> **TODO.** Transpose `consensus`, attach `target_sets`, and run
> `pt.tl.Enrichment().score(...)`. Sanity-check on `TUBB`/`TUBA4A`, which should score
> highly given the positive controls.


## 9. What does not translate

| `pertpy` tool | why it does not apply here | when it would |
| --- | --- | --- |
| `Mixscape`, `Mixscale` | need single cells with assigned gRNAs, to separate perturbed from escaping cells | single-cell morphology from a CRISPR Cell Painting screen |
| `Augur` | prioritises **cell types** by perturbation responsiveness | a dataset with annotated cell types, or single-cell profiles clustered into states |
| `Milo` | differential abundance of neighbourhoods across samples | single-cell morphology; well-level data has no within-sample composition |
| `Sccoda`, `Tasccoda` | compositional analysis of cell-type proportions | single-cell morphology binned into states |
| `Scgen`, `Cinemaot` | predict/deconvolve perturbation response in expression space | requires expression; morphology has no equivalent generative model |
| `PyDESeq2`, `EdgeR` | negative-binomial **count** models | never on z-scored profiles — use `TTest`/`WilcoxonTest` instead |
| dose-response tooling | needs a dose series | EUbOPEN, or a JUMP dose-titration subset |

The pattern: everything that needs **single cells** or **counts** is out; everything that
compares **groups of observations in a continuous space** is in. Most of what is
unavailable becomes available if you profile single cells rather than well medians —
worth remembering, since the underlying CellProfiler output usually *is* per-cell before
aggregation.

> **The most valuable extension** would be re-running this on single-cell profiles.
> `Mixscape`-style analysis would then let you ask what fraction of cells in a well
> actually responded — a question well-level medians cannot even pose, and one that
> matters because a median hides whether 10% of cells responded strongly or 100%
> responded weakly.


## Key takeaways

1. **A Cell Painting screen is a perturbation experiment**, so pertpy's group-comparison
   tools apply directly — as long as you work in PCA space and respect the sample sizes.
2. **n = 4 wells per compound is the binding constraint.** Distribution metrics
   (`edistance`, `mmd`, `wasserstein`) are noisy at that size; mean-based metrics
   (`mse`, `pearson_distance`) are cheaper and better behaved.
3. **No dose-response in this dataset** — concentration is collinear with `pert_type`.
   Switch to EUbOPEN (10 doses) if you need it.
4. **Replicate reproducibility is the metric that matters**, has no single-cell analogue,
   and is not in pertpy. Implement it, and **state your null** — percent replicating swings
   from ~82% to ~65% depending on whether the null controls for well position. Combine it
   with induction to call hits.
5. **Use `mode="median"` for consensus profiles.** The `sum` default is a count-data
   assumption and is wrong for z-scores.
6. **Positive controls are essential and imperfect.** Tubulin binders recover the assay,
   but the annotation includes EGFR inhibitors — inspect before trusting any label.
7. **Everything requiring single cells or counts is out of scope** — and would come back
   into scope with single-cell morphological profiles.

## References

- [pertpy documentation](https://pertpy.readthedocs.io/) — Heumos et al., https://doi.org/10.1101/2024.08.04.606516
- [Perturbation modelling chapter, single-cell best practices](https://www.sc-best-practices.org/conditions/perturbation_modeling.html)
- Peidli et al. 2024, *scPerturb* (E-distance for perturbation effects) — https://doi.org/10.1038/s41592-023-02144-y
- [`copairs`](https://github.com/cytomining/copairs) — reference mAP implementation for profiling
- Way et al. 2021, *Predicting cell health phenotypes* — https://doi.org/10.1091/mbc.E20-12-0784